# MoE — 一个模型里住着 8 个专家

> **背景**：GPT-4 据说有 1.8 万亿参数，但每次推理只激活其中一部分。怎么做到的？
>
> 本 Part 目标：**理解 Mixture of Experts (MoE) 的核心原理——把 FFN 层换成多个「专家」，每次只激活其中几个。**

核心思想一句话：**传统 Transformer 的 FFN 层是一个「全科医生」，MoE 把它换成 8 个「专科医生」，每个 token 只找最对口的 2 个医生看。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

### 1. 回顾：普通 Transformer 的 FFN 层

每个 Transformer Block 里有一个 FFN（前馈网络）：

```
输入 x (d_model=512)
  ↓
Linear(512 → 2048)  ← 升维
  ↓
ReLU / GELU
  ↓
Linear(2048 → 512)  ← 降维
  ↓
输出
```

参数量：512×2048 + 2048×512 ≈ 2M

**问题**：所有 token 都经过同一个 FFN。不管你是「的」还是「量子力学」，都用同一套参数处理。
要让模型更聪明，就得把 FFN 变大 → 参数爆炸。

### 2. MoE 的核心思想：多个专家 + 路由器

把一个大 FFN 拆成 N 个小 FFN（专家），加一个「路由器」决定每个 token 找哪几个专家。

```
               ┌─────────────┐
               │   路由器     │  ← 决定找谁
               │  (Gate)     │
               └──┬──┬──┬───┘
                  │  │  │
          ┌───────┘  │  └───────┐
          ↓          ↓          ↓
     ┌────────┐┌────────┐┌────────┐
     │ 专家 1 ││ 专家 2 ││ 专家 3 │  ... (共 8 个)
     │ (FFN)  ││ (FFN)  ││ (FFN)  │
     └────────┘└────────┘└────────┘
          ↓          ↓          ↓
          └──────────┴──────────┘
                    加权求和
```

**关键**：每个 token 只激活 top-k 个专家（通常 k=2），不是全部 8 个。

```
token "的"     → 路由器 → 专家 1, 5  (简单 token，随便找)
token "量子"   → 路由器 → 专家 3, 7  (物理相关)
token "hello"  → 路由器 → 专家 2, 6  (英文相关)
```

**效果**：
- 总参数量 = 8 × 一个专家的参数量（很大！）
- 每次推理的计算量 = 2 × 一个专家的参数量（和普通 FFN 差不多！）
- **参数多但计算少** ← 这就是 MoE 的魔法

In [ ]:
class MoELayer(nn.Module):
    """
    MoE FFN 层
    
    参数:
        d_model: 隐藏维度
        num_experts: 专家数量
        top_k: 每个 token 激活几个专家
    """
    def __init__(self, d_model, num_experts=8, top_k=2, expert_dim=None):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        expert_dim = expert_dim or 4 * d_model
        
        # 路由器：输入 d_model，输出 num_experts 个分数
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        
        # N 个专家，每个是一个小 FFN
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, d_model)
            )
            for _ in range(num_experts)
        ])
    
    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        
        流程:
        1. 路由器给每个 token 打分
        2. 选 top-k 个专家
        3. 只算这 k 个专家的输出
        4. 加权求和
        """
        batch_size, seq_len, d_model = x.shape
        
        # Step 1: 路由器打分
        gate_logits = self.gate(x)  # [batch, seq_len, num_experts]
        
        # Step 2: 选 top-k
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # 归一化权重
        
        # Step 3 & 4: 对每个 token，算选中专家的输出并加权求和
        output = torch.zeros_like(x)
        
        for b in range(batch_size):
            for s in range(seq_len):
                token = x[b, s]  # [d_model]
                for k in range(self.top_k):
                    expert_idx = top_k_indices[b, s, k].item()
                    weight = top_k_weights[b, s, k]
                    expert_out = self.experts[expert_idx](token.unsqueeze(0)).squeeze(0)
                    output[b, s] += weight * expert_out
        
        return output

print("MoE 层定义完成！")
print(f"8 个专家，每个 token 只激活 2 个")

In [ ]:
# 演示 MoE 的路由过程
moe = MoELayer(d_model=16, num_experts=8, top_k=2)

# 模拟 3 个 token
x = torch.randn(1, 3, 16)

# 看路由器给每个 token 的评分
with torch.no_grad():
    gate_scores = moe.gate(x).squeeze(0)  # [3, 8]
    top_k_vals, top_k_idx = torch.topk(gate_scores, 2, dim=-1)

print("=== 路由器为 3 个 token 选择的专家 ===")
print()
for i in range(3):
    experts = top_k_idx[i].tolist()
    weights = F.softmax(top_k_vals[i], dim=-1).tolist()
    print(f"Token {i}: 选中专家 {experts}, 权重 {[f'{w:.2f}' for w in weights]}")

print()
print("每个 token 只激活 2/8 = 25% 的专家")
print("总参数是 8 个专家的和，但计算量只有 2 个专家的量")

### 3. MoE 的参数 vs 计算量

这是 MoE 最核心的优势：

```
普通 Dense 模型:
  FFN 参数: 2M
  每次计算: 2M 参数都参与
  
  模型变大 → 参数和计算量同步增长

MoE 模型 (8 专家, top-2):
  FFN 参数: 8 × 2M = 16M  ← 参数多了 8 倍！
  每次计算: 2 × 2M = 4M   ← 计算只多了 2 倍
  
  参数和计算量「解耦」了！
```

这就是为什么 Mixtral 8×7B（MoE）只有 7B 的计算量，却有 47B 的「知识容量」。

### 4. MoE 的训练难题：负载均衡

MoE 有一个天生的坑：**路由器可能偷懒，只把 token 发给少数几个专家。**

```
坏情况（负载不均）:
  专家 1: ████████████████████  (忙死)
  专家 2: ████
  专家 3: █
  专家 4-8: (闲死，参数白训了)

好情况（负载均衡）:
  专家 1: ██████
  专家 2: ██████
  专家 3: ██████
  ...
  专家 8: ██████
```

**解决方案**：加一个辅助 loss，惩罚负载不均。

```python
# 负载均衡 loss（简化版）
# 鼓励每个专家处理的 token 数尽量均匀

load_balance_loss = 0
for expert_i in range(num_experts):
    # 这个专家实际处理了多少 token
    actual_load = count_tokens_assigned_to(expert_i)
    # 理想情况下每个专家应该处理多少
    ideal_load = total_tokens * top_k / num_experts
    load_balance_loss += (actual_load - ideal_load) ** 2
```

### 5. MoE 的推理难题：所有专家都要加载

虽然每次只激活 2 个专家，但**你不知道哪 2 个会被激活**，所以所有 8 个专家的参数都得在显存里。

```
Dense 7B:  显存 ≈ 14GB (FP16)
MoE 8×7B:  显存 ≈ 94GB (FP16) ← 8 倍！
```

这就是为什么 MoE 模型虽然「推理快」（计算量小），但「显存大」（参数多）。

**解决方案**：
- 量化（INT4/INT8）压缩参数
- 把不常用的专家 offload 到 CPU 内存
- 多 GPU 分布式部署

### 6. 著名的 MoE 模型

| 模型 | 总参数 | 激活参数 | 专家数 | Top-K |
|------|--------|---------|--------|-------|
| **Mixtral 8×7B** | 47B | 13B | 8 | 2 |
| **DeepSeek-V2** | 236B | 21B | 160 | 6 |
| **GPT-4 (传闻)** | 1.8T | ~280B | 8-16 | 2 |
| **Qwen2.5-MoE** | 57B | 14B | 64 | 8 |

**关键数字**：激活参数 / 总参数 ≈ 10-20%。
也就是说，MoE 模型 80-90% 的参数在每次推理时是「休眠」的。

### 7. MoE 的直觉类比

```
普通 Dense 模型 = 一个全科医生
  - 什么病都能看
  - 但每个病都看得不深
  - 要变强只能让这个医生学更多 → 更慢

MoE 模型 = 一个医院，有 8 个专科医生
  - 心脏科、脑科、骨科...
  - 每个病人只看对口的 2 个医生
  - 医院总人数多（参数多），但每个病人等待时间短（计算少）
  
  路由器 = 分诊台护士
  "你头疼 → 找神经科 + 内科"
  "你腿疼 → 找骨科 + 康复科"
```

### 8. MoE 的进阶话题

**DeepSeek-V2 的创新**：
- 传统 MoE 用「硬路由」（选 top-k，其他权重为 0）
- DeepSeek 用「软路由」+「共享专家」
- 共享专家：所有 token 都经过的基础专家（处理通用知识）
- 路由专家：top-k 激活的专家（处理专业知识）

```
DeepSeek MoE:
  token → 共享专家（必过） + 路由专家 top-k（选过）
  
  共享专家处理: 语法、常识
  路由专家处理: 数学、编程、翻译...
```

**Expert Parallelism**：
- 不同专家放在不同 GPU 上
- token 被路由器分发到对应 GPU
- 需要 all-to-all 通信（MoE 的主要开销）

---

## MoE 支线小结

1. ✅ 普通 FFN 是「全科医生」，所有 token 用同一套参数
2. ✅ MoE 把 FFN 换成 N 个「专科医生」+ 一个「分诊台」（路由器）
3. ✅ 每个 token 只激活 top-k 个专家 → 参数多但计算少
4. ✅ 参数和计算量「解耦」→ 可以用更少的计算获得更多的知识
5. ✅ 训练难题：负载均衡 → 加辅助 loss
6. ✅ 推理难题：所有专家都要在显存里 → 量化 / 多 GPU
7. ✅ 著名 MoE：Mixtral 8×7B、DeepSeek-V2、GPT-4
8. ✅ 激活参数 ≈ 总参数的 10-20%

**一句话总结**：MoE = 一个模型里住着 8 个专家，路由器决定每个 token 找哪 2 个。参数多了 8 倍，计算只多了 2 倍。